# DocuLens — Layer 1: PDF Ingestion

Run this notebook in **Google Colab** to test the ingestion layer without needing to install anything locally.

**What this does:**
1. Installs all dependencies
2. Clones your GitHub repo (or uploads files directly)
3. Generates synthetic German invoices for testing
4. Runs the PDF router on them
5. Shows extraction results
6. Runs the test suite

## Step 1 — Install dependencies

In [ ]:
# Core PDF and OCR libraries
!pip install -q pymupdf pdf2image pytesseract reportlab Pillow

# Install Tesseract with German language pack
!apt-get install -q tesseract-ocr tesseract-ocr-deu poppler-utils

print('All dependencies installed!')

## Step 2 — Upload your project files

Either clone from GitHub or upload manually.

In [ ]:
# Option A: Clone from GitHub (once you push your repo)
# !git clone https://github.com/YOUR_USERNAME/doculens.git
# %cd doculens

# Option B: Upload files directly in Colab
# from google.colab import files
# uploaded = files.upload()

# Option C: Copy-paste the code directly below (for quick testing)
import sys, os
os.makedirs('src/ingestion', exist_ok=True)
os.makedirs('src/utils', exist_ok=True)
os.makedirs('data/samples', exist_ok=True)

# Write empty __init__ files
for path in ['src/__init__.py', 'src/ingestion/__init__.py', 'src/utils/__init__.py']:
    open(path, 'w').close()

print('Project structure ready.')

## Step 3 — Generate synthetic German invoices

In [ ]:
# Add project root to path
sys.path.insert(0, '.')

from src.utils.generate_samples import generate_batch

print('Generating 5 synthetic German invoices...')
paths = generate_batch('data/samples', n=5)
print(f'\nGenerated {len(paths)} PDFs:')
for p in paths:
    print(f'  {p}')

## Step 4 — Run the PDF router

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

from src.ingestion.pdf_router import PDFRouter

router = PDFRouter(ocr_lang='deu+eng')

# Process all generated invoices
results = []
for path in paths:
    result = router.process(path)
    results.append(result)
    print(f'\n--- {path.name} ---')
    print(f'  Type   : {result.doc_type.value}')
    print(f'  Method : {result.extraction_method.value}')
    print(f'  Pages  : {result.page_count}')
    print(f'  Chars  : {len(result.full_text)}')
    print(f'  Success: {result.success}')

## Step 5 — Inspect extracted text

In [ ]:
# Look at the first result in detail
r = results[0]
print(f'=== {r.file_path} ===')
print(f'Document type  : {r.doc_type.value}')
print(f'Extraction     : {r.extraction_method.value}')
print(f'Page count     : {r.page_count}')
print()
print('--- First 800 characters of extracted text ---')
print(r.full_text[:800])

## Step 6 — Run the test suite

In [ ]:
!pip install -q pytest
!pytest tests/test_ingestion.py -v --tb=short 2>&1

## Step 7 — Upload a real German PDF (optional)

Try it on a real document!

In [ ]:
from google.colab import files
import io

print('Upload a German PDF to test:')
uploaded = files.upload()

for filename, content in uploaded.items():
    # Save to disk
    with open(filename, 'wb') as f:
        f.write(content)

    # Process it
    result = router.process(filename)
    print(f'\nFile      : {filename}')
    print(f'Type      : {result.doc_type.value}')
    print(f'Method    : {result.extraction_method.value}')
    print(f'Pages     : {result.page_count}')
    print(f'Characters: {len(result.full_text)}')
    if result.avg_ocr_confidence:
        print(f'OCR conf  : {result.avg_ocr_confidence}%')
    print()
    print('--- Preview (first 600 chars) ---')
    print(result.full_text[:600])